# Visualize samples with the TransFuser visualizer

The dataset root resolves from the repo's `.env`; the visualizer consumes a collated batch, as during training.

In [ ]:
import os

from lead.common.env import read_dotenv

os.environ.setdefault(
    "PY123D_DATA_ROOT", read_dotenv("PY123D_DATA_ROOT", "data/lead/123D")
)

In [ ]:
from lead.config import load_lead_config

config = load_lead_config()
config.training.experiment.visualize_dataset = True

In [ ]:
from lead.policy.transfuser.transfuser import Transfuser

dataset = Transfuser(config).build_dataset()
print(f"{len(dataset)} samples")
sample = dataset.collate_fn([dataset[0]])

In [ ]:
from PIL import Image

from lead.policy.transfuser.visualization.ground_truth_visualizer import (
    GroundTruthVisualizer,
)

image = GroundTruthVisualizer(config, data=sample).visualize()
Image.fromarray(image)

## Render the whole scene

In [ ]:
import subprocess

from IPython.display import Video

encoder = subprocess.Popen(
    [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-f",
        "rawvideo",
        "-pix_fmt",
        "rgb24",
        "-s",
        f"{image.shape[1]}x{image.shape[0]}",
        "-framerate",
        "4",
        "-i",
        "-",
        "-vf",
        "crop=trunc(iw/2)*2:trunc(ih/2)*2",
        "-pix_fmt",
        "yuv420p",
        "scene.mp4",
    ],
    stdin=subprocess.PIPE,
)
for i in range(len(dataset)):
    frame = GroundTruthVisualizer(
        config, data=dataset.collate_fn([dataset[i]])
    ).visualize()
    encoder.stdin.write(frame.tobytes())
encoder.stdin.close()
assert encoder.wait() == 0
Video("scene.mp4", width=800)